## Objectif : compter le nombre d'écoles à proximité de chaque station et voir si cela influe le nombre de validations de titres.

# Imports.

In [ ]:
import os
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# Télécharger les positions des écoles en Ile-de-France.

In [ ]:
# Objectif : compter le nombre d'écoles à proximité de chaque station et voir si cela influe le nombre de validations de titres Imagine R.

url_ecoles = "https://data.iledefrance.fr/api/explore/v2.1/catalog/datasets/les_etablissements_d_enseignement_des_1er_et_2d_degres_en_idf/exports/csv?lang=fr&timezone=Europe%2FBerlin&use_labels=true&delimiter=%3B"

# Chemin du répertoire pour mettre le fichier csv d'emplacement des écoles.
folder_path_ecoles = os.path.join(
    "..", "data", "raw"
)

# Vérifier et créer le répertoire de destination s'il n'existe pas.

if not os.path.exists(folder_path_ecoles):
    os.makedirs(folder_path_ecoles)
    print(f"Répertoire créé : {folder_path_ecoles}")

In [ ]:
# Définir une fonction pour télécharger les sources.

def telecharger_csv(url, destination, fichier):
    # Chemin complet du fichier.
    filepath = os.path.join(destination, fichier)
    try:
        print(f"Téléchargement en cours depuis : {url}")

        # Effectuer une requête HTTP.
        reponse = requests.get(url, timeout=10)

        # Erreur si le statut HTTP n'est pas bon (404, 500).
        reponse.raise_for_status()

        # Ecrire le contenu dans un fichier (en mode binaire).
        with open(filepath, "wb") as f:
            f.write(reponse.content)

        print(f"Fichier enregistré sous : {filepath}")

    except requests.exceptions.RequestException as e:
        print(f"Erreur lors du téléchargement : {e}")


In [ ]:
if __name__ == "__main__":
    # Télécharger le fichier des emplacements des écoles.

    url = url_ecoles
    destination = folder_path_ecoles
    fichier = "ecoles.csv"

    telecharger_csv(url, destination, fichier)

# Explorer et nettoyer les données des écoles.

In [ ]:
filepath_ecoles = os.path.join(folder_path_ecoles, "ecoles.csv")
ecoles_df = pd.read_csv(filepath_ecoles, sep=";")

In [ ]:
ecoles_df.head()

In [ ]:
ecoles_df.shape

In [ ]:
ecoles_df.columns

In [ ]:
# Supprimer les colonnes non utiles.
colonnes_supprimer = [
    "Numéro d'UAI",
    'Adresse : désignation de la voie', 
    'Adresse : 5e ligne',
    'Adresse : boite postale ou course spéciale', 
    "Localité d'acheminement",
    'Libellé de la commune',
    'Géolocalisation : coordonnée X', 
    'Géolocalisation : coordonnée Y',
    'EPSG', 
    'Appariement par IGN',
    'localisation par IGN', 
    "Code nature de l'UAI",
    "Libellé de la nature de l'UAI",
    "Etat de l'établissement",
    "Libellé de l'état de l'établissement",
    'Code INSEE du département ou de la collectivité',
    'Code INSEE de la région', 
    "Code de l'académie",
    'Code INSEE de la commune',
    'Libellé du département ou de la collectivité', 
    'Libellé de la région',
    "Libellé de l'académie", 
    'Code du type de contrat', 'Libellé du type de contrat',
    'Code de la tutelle ministérielle', 
    'Libellé de la tutelle',
    'Date de rentrée pédagogique', 
    "Sigle de l'UAI", 
    'Identifiants RNB',
    'Latitude et longitude WGS84'
    
]

ecoles_df = ecoles_df.drop(columns=colonnes_supprimer)

In [ ]:
ecoles_df.head()

In [ ]:
# Normaliser les noms de colonnes.
ecoles_df.columns = (
    ecoles_df.columns.str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
    .str.lower()
    .str.replace(" ", "_")
)

In [ ]:
ecoles_df.head()

In [ ]:
ecoles_df.info()

In [ ]:
ecoles_df.describe()

In [ ]:
ecoles_df.isna().sum()

In [ ]:
# Suppimer les valeurs manquantes qui correspondent à latitude_wgs84 et longitude_wgs84.
ecoles_df = ecoles_df.dropna(subset=["latitude_wgs84"])

In [ ]:
ecoles_df.isna().sum()

In [ ]:
ecoles_df.shape

# Données liste des stations.

In [ ]:
filepath = os.path.join("..", "data", "processed", "stations.csv")
stations_df = pd.read_csv(filepath)

In [ ]:
stations_df.head()

# Cross-join avec GeoPandas.

In [ ]:
# Convertir les DataFrames stations et écoles en GeoDataFrames.
gdf_stations = gpd.GeoDataFrame(
    stations_df,
    geometry=gpd.points_from_xy(stations_df['longitude'], stations_df['latitude']),
    crs="EPSG:4326"
)

gdf_ecoles = gpd.GeoDataFrame(
    ecoles_df,
    geometry=gpd.points_from_xy(ecoles_df['longitude_wgs84'], ecoles_df['latitude_wgs84']),
    crs="EPSG:4326"
)


In [ ]:
# Projeter en Lambert-93 (EPSG:2154) pour travailler en mètre.
gdf_stations = gdf_stations.to_crs(epsg=2154)
gdf_ecoles = gdf_ecoles.to_crs(epsg=2154)


In [ ]:
# Créer un rayon de recherche autour des écoles.
distance_max = 500 # en mètres.
gdf_ecoles['zone_recherche'] = gdf_ecoles.geometry.buffer(distance_max)

In [ ]:
# Définir le buffer comme la géométrie active pour la jointure.
gdf_ecoles_buffer = gdf_ecoles.set_geometry('zone_recherche')

In [ ]:
# Jointure spatiale pour trouver les écoles qui sont dans le rayon de la station.
ecoles_stations_df = gpd.sjoin(
    gdf_stations,
    gdf_ecoles_buffer[['appellation_officielle', 'secteur', 'zone_recherche']],
    how='inner',
    predicate='within'
)

In [ ]:
ecoles_stations_df.head()

In [ ]:
ecoles_stations_df.shape

In [ ]:
# Test de filtrage par nom école.
ecoles_stations_df[ecoles_stations_df['appellation_officielle'].str.contains('Stanislas', na=False)]

In [ ]:
# Test de filtrage par nom station.
ecoles_stations_df[ecoles_stations_df['nom_zdc'].str.contains("Pasteur", na=False)]

In [ ]:
# Création de nouvelle colonne nombre d'écoles par station.
ecoles_stations_df['nb_ecoles'] = ecoles_stations_df.groupby("nom_zdc")['nom_zdc'].transform('size')

In [ ]:
ecoles_stations_df.head()

In [ ]:
# Ne garder que 3 colonnes.
ecoles_stations_df = ecoles_stations_df[["id_ref_zdc", "nom_zdc", "nb_ecoles"]]

In [ ]:
ecoles_stations_df.head(15)

In [ ]:
# Suppression des doublons.
ecoles_stations_df = ecoles_stations_df.drop_duplicates()

In [ ]:
ecoles_stations_df.head(10)

In [ ]:
ecoles_stations_df.shape

In [ ]:
# Test.
ecoles_stations_df[ecoles_stations_df["nom_zdc"].str.contains("Pasteur", na=False)]

# Merge avec validations_fusion.

In [ ]:
filepath = os.path.join("..", "data", "processed", "validations_fusion.csv")
validations_fusion_df = pd.read_csv(filepath)

In [ ]:
validations_fusion_df.head()

In [ ]:
# Merge
df = validations_fusion_df.merge(
    ecoles_stations_df, how="left", left_on="id_zdc", right_on="id_ref_zdc"
)

In [ ]:
df.head()

In [ ]:
# Supprimer colonnes en doublon.
colonnes_supprimer_merge = [
    "id_ref_zdc_y",
    "nom_zdc_y"
]
df = df.drop(columns=colonnes_supprimer_merge)

In [ ]:
# Renommer les colonnes.
df.rename(
    columns={
        "id_ref_zdc_x": "id_ref_zdc",
        "nom_zdc_x": "nom_zdc"
    }, inplace=True
)

In [ ]:
# Remplacer les valeurs NaN de "nb_ecoles" par zéro.
df["nb_ecoles"] = df["nb_ecoles"].fillna(0)

In [ ]:
# Changer nb_ecoles de float à int.
df["nb_ecoles"] = df["nb_ecoles"].astype(int)

In [ ]:
df.head()

In [ ]:
# Créer une colonne pourcentage validations Imagine R par rapport à la validation totale par station.

# Validations Imagine R par station.
imagine_r_par_station = df[df["categorie_titre"] == "Imagine R"].groupby("nom_zdc")["nb_vald"].sum()

In [ ]:
imagine_r_par_station.head()

In [ ]:
# Validations totales par station.
total_par_station = df.groupby("nom_zdc")["nb_vald"].sum()

In [ ]:
total_par_station.head()

In [ ]:
# Combinaison dans un nouveau DataFrame.
part_imagine_r_df = pd.DataFrame({
    "nb_vald": total_par_station,
    "nb_vald_imagine_r": imagine_r_par_station,
    "pourcent_vald_imagine_r": (imagine_r_par_station / total_par_station * 100).round(2)
}).reset_index()

In [ ]:
part_imagine_r_df.head()

In [ ]:
# Test
part_imagine_r_df[part_imagine_r_df["nom_zdc"].str.contains("Pasteur", na=False)]

In [ ]:
ecoles_stations_df = df.groupby("nom_zdc").agg({"nb_ecoles": "max", "nb_vald": "sum"}).reset_index()

In [ ]:
ecoles_stations_df.head()

In [ ]:
fusion_ecoles_df = ecoles_stations_df.merge(
    part_imagine_r_df, how="left", left_on="nom_zdc", right_on="nom_zdc"
)

In [ ]:
fusion_ecoles_df.head()

In [ ]:
# Supprimer colonnes en doublon.
colonnes_supprimer_merge = [
    "nb_vald_y"
]
fusion_ecoles_df = fusion_ecoles_df.drop(columns=colonnes_supprimer_merge)

In [ ]:
# Renommer une colonne.
fusion_ecoles_df.rename(
    columns={
        "nb_vald_x": "nb_vald"
    }, inplace=True
)

In [ ]:
fusion_ecoles_df.head()

In [ ]:
# Test
fusion_ecoles_df[fusion_ecoles_df["nom_zdc"].str.contains("Pasteur", na=False)]

In [ ]:
# Corrélation entre nombre d'écoles à proximité des stations et le nombre de validation de titres Imagine R.

# Graphique : Validations de titres Imagin R par nombre d'écoles à proximité.
plt.figure(figsize=(10,5))

plt.scatter(
    x="nb_ecoles",
    y="nb_vald_imagine_r",
    data= fusion_ecoles_df,
    marker='o',
    alpha=0.5
)
plt.title("Validations de titres Imagin R par nombre d'écoles à proximité.")
plt.xlabel("Nombre d'écoles à proximité")
plt.ylabel("Validations de titre Imagin R")
plt.show()

In [ ]:
# Corrélation entre le nombre d'écoles à proximité des stations et la part de validation de titres Imagine R sur le nombre de validations totales.

# Graphique : Part de validations de titres Imagin R par nombre d'écoles à proximité.
plt.figure(figsize=(10,5))

plt.scatter(
    x="nb_ecoles",
    y="pourcent_vald_imagine_r",
    data=fusion_ecoles_df,
    marker='o',
    alpha=0.5
)
plt.title("Part de validations de titres Imagine R par nombre d'écoles à proximité.")
plt.xlabel("Nombre d'écoles à proximité")
plt.ylabel("% Validations de titre Imagine R")
plt.show()

# Conclusion

Il n'y a pas de corrélation : 

entre le nombre d'écoles accessibles par station (500m) et le nombre de validations de titre de transport "Imagine R".

entre le nombre d'écoles accessibles par station (500m) et la part de validations de titre de transport "Imagine R" sur le nombre de validations totales.